In [1]:
import requests
import json
import logging
from datetime import datetime
from kafka import KafkaProducer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

def fetch_and_produce():
    url = "https://api.open-meteo.com/v1/forecast?latitude=47.9&longitude=106.9&current_weather=true"
    response = requests.get(url)
    data = response.json()["current_weather"]
    data["fetched_at"] = datetime.utcnow().isoformat()
    producer.send("weather_stream", data)
    producer.flush()
    logger.info(f"Sent: {data}")

fetch_and_produce()

2026-05-08 22:32:03,495 - INFO - <BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: connecting to localhost:9092 [('::1', 9092, 0, 0) IPv6]
2026-05-08 22:32:03,503 - INFO - <BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <checking_api_versions_recv> [IPv6 ('::1', 9092, 0, 0)]>: Broker version identified as 2.6
2026-05-08 22:32:03,504 - INFO - <BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Connection complete.
/var/folders/hv/wjvkzpmj6fv44k0l243tmj_h0000gn/T/ipykernel_4443/3588509288.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  data["fetched_at"] = datetime.utcnow().isoformat()
2026-05-08 22:32:03,992 - INFO - 

In [9]:
response = requests.get("https://api.open-meteo.com/v1/forecast?latitude=47.9&longitude=106.9&current_weather=true")
print(response.status_code)
print(response.text)

200
{"latitude":47.90861,"longitude":106.86567,"generationtime_ms":0.12767314910888672,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":1287.0,"current_weather_units":{"time":"iso8601","interval":"seconds","temperature":"°C","windspeed":"km/h","winddirection":"°","is_day":"","weathercode":"wmo code"},"current_weather":{"time":"2026-05-08T14:30","interval":900,"temperature":7.6,"windspeed":8.9,"winddirection":339,"is_day":0,"weathercode":0}}
